In [1]:
import numpy as np

def get_sgp_mat(num_in, num_out, link):
    A = np.zeros((num_in, num_out))
    for i, j in link:
        A[i, j] = 1
    A_norm = A / np.sum(A, axis=0, keepdims=True)
    return A_norm

def edge2mat(link, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in link:
        A[j, i] = 1
    return A

def get_k_scale_graph(scale, A):
    if scale == 1:
        return A
    An = np.zeros_like(A)
    A_power = np.eye(A.shape[0])
    for k in range(scale):
        A_power = A_power @ A
        An += A_power
    An[An > 0] = 1
    return An

def normalize_digraph(A):
    Dl = np.sum(A, 0)
    h, w = A.shape
    Dn = np.zeros((w, w))
    for i in range(w):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** (-1)
    AD = np.dot(A, Dn)
    return AD


def get_spatial_graph(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node)
    In = normalize_digraph(edge2mat(inward, num_node))
    Out = normalize_digraph(edge2mat(outward, num_node))
    A = np.stack((I, In, Out))
    return A

def get_ins_spatial_graph(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node)
    In = normalize_digraph(np.ones_like(I)-I)
    Out = In.T
    A = np.stack((I, In, Out))
    return A

def get_spatial_graphnextv2(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node)
    #In = normalize_digraph(edge2mat(inward, num_node))
    #Out = normalize_digraph(edge2mat(outward, num_node))
    #SELF = np.eye(num_node)
    A = np.stack((I, I, I, I))
    return A

def get_spatial_graphnext(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node)
    In = normalize_digraph(edge2mat(inward, num_node))
    Out = normalize_digraph(edge2mat(outward, num_node))
    SELF = np.eye(num_node)
    A = np.stack((I, In, Out, SELF))
    return A

def normalize_adjacency_matrix(A):
    node_degrees = A.sum(-1)
    degs_inv_sqrt = np.power(node_degrees, -0.5)
    norm_degs_matrix = np.eye(len(node_degrees)) * degs_inv_sqrt
    return (norm_degs_matrix @ A @ norm_degs_matrix).astype(np.float32)


def k_adjacency(A, k, with_self=False, self_factor=1):
    assert isinstance(A, np.ndarray)
    I = np.eye(len(A), dtype=A.dtype)
    if k == 0:
        return I
    Ak = np.minimum(np.linalg.matrix_power(A + I, k), 1) \
       - np.minimum(np.linalg.matrix_power(A + I, k - 1), 1)
    if with_self:
        Ak += (self_factor * I)
    return Ak

def get_multiscale_spatial_graph(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node)
    A1 = edge2mat(inward, num_node)
    A2 = edge2mat(outward, num_node)
    A3 = k_adjacency(A1, 2)
    A4 = k_adjacency(A2, 2)
    A1 = normalize_digraph(A1)
    A2 = normalize_digraph(A2)
    A3 = normalize_digraph(A3)
    A4 = normalize_digraph(A4)
    A = np.stack((I, A1, A2, A3, A4))
    return A



def get_uniform_graph(num_node, self_link, neighbor):
    A = normalize_digraph(edge2mat(neighbor + self_link, num_node))
    return A

num_node = 25

self_link = [(i, i) for i in range(num_node)]

inward = [
    (1, 0), (2, 1), (3, 2), (4, 3),
    (5, 1), (6, 5), (7, 6),
    (8, 1), (9, 8), (10, 9),
    (11, 8), (12, 11), (13, 12),
    (14, 0), (15, 0),
    (16, 14), (17, 15),
    (18, 14), (19, 18), (20, 19),
    (21, 14), (22, 11),
    (23, 22), (24, 11)
]

outward = [(j, i) for (i, j) in inward]

neighbor = inward + outward

class Graph:

    def __init__(self, labeling_mode='spatial'):

        self.num_node = num_node
        self.self_link = self_link
        self.inward = inward
        self.outward = outward
        self.neighbor = neighbor

        self.A = self.get_adjacency_matrix(labeling_mode)

    def get_adjacency_matrix(self, labeling_mode=None):

        if labeling_mode is None:
            return self.A

        if labeling_mode == 'spatial':
            A = get_spatial_graph(
                self.num_node,
                self.self_link,
                self.inward,
                self.outward
            )

        else:
            raise ValueError("Unsupported labeling mode")

        return A

In [2]:
import math
import pdb
from termios import VINTR

import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F
from timm.models.layers import trunc_normal_, DropPath
from timm.models.registry import register_model
from torch import Tensor
from typing import Tuple
def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod

class PositionEmbeddingSine(nn.Module):
    def __init__(self, numPositionFeatures: int = 64, temperature: int = 10000, normalize: bool = True,
                 scale: float = None):
        super(PositionEmbeddingSine, self).__init__()

        self.numPositionFeatures = numPositionFeatures
        self.temperature = temperature
        self.normalize = normalize

        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor]:
        N, _, H, W = x.shape

        mask = torch.zeros(N, H, W, dtype=torch.bool, device=x.device)
        notMask = ~mask

        yEmbed = notMask.cumsum(1)
        xEmbed = notMask.cumsum(2)

        if self.normalize:
            epsilon = 1e-6
            yEmbed = yEmbed / (yEmbed[:, -1:, :] + epsilon) * self.scale
            xEmbed = xEmbed / (xEmbed[:, :, -1:] + epsilon) * self.scale

        dimT = torch.arange(self.numPositionFeatures, dtype=torch.float32, device=x.device)
        dimT = self.temperature ** (2 * (dimT // 2) / self.numPositionFeatures)

        posX = xEmbed.unsqueeze(-1) / dimT
        posY = yEmbed.unsqueeze(-1) / dimT

        posX = torch.stack((posX[:, :, :, 0::2].sin(), posX[:, :, :, 1::2].cos()), -1).flatten(3)
        posY = torch.stack((posY[:, :, :, 0::2].sin(), posY[:, :, :, 1::2].cos()), -1).flatten(3)

        return torch.cat((posY, posX), 3).permute(0, 3, 1, 2)
class LayerNorm(nn.Module):
    r""" LayerNorm that supports two data formats: channels_last (default) or channels_first. 
    The ordering of the dimensions in the inputs. channels_last corresponds to inputs with 
    shape (batch_size, height, width, channels) while channels_first corresponds to inputs 
    with shape (batch_size, channels, height, width).
    """
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ["channels_last", "channels_first"]:
            raise NotImplementedError 
        self.normalized_shape = (normalized_shape, )
    
    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x


class TSGCNext_unit(nn.Module):
    def __init__(self, in_channels, out_channels, A, coff_embedding=4, adaptive=True, residual=True,layer_scale_init_value=1e-6,drop_path=0.):
        super(TSGCNext_unit, self).__init__()
        inter_channels = out_channels // coff_embedding
        self.inter_c = inter_channels
        self.out_c = out_channels
        self.in_c = in_channels
        self.adaptive = adaptive
        if residual:
            self.dwconv = nn.Conv2d(in_channels, in_channels, kernel_size=(3,1), padding=(1,0), groups=in_channels) # depthwise conv Twise
        else:
            self.dwconv = nn.Conv2d(in_channels, out_channels, kernel_size=(4,1), stride=(4,1))
            in_channels = out_channels
        self.norm = LayerNorm(in_channels, eps=1e-6)
        self.pwconv1 = nn.Linear(in_channels, 4 * in_channels) # pointwise/1x1 convs, implemented with linear layers
        self.act = nn.GELU()
       
        self.PA = nn.Parameter(torch.from_numpy(A.astype(np.float32)))
        

        self.pwconv2 = nn.Linear(4 * in_channels, in_channels)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones((in_channels)), 
                                    requires_grad=True) if layer_scale_init_value > 0 else None
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        if in_channels != out_channels:
            self.down = nn.Sequential(
                LayerNorm(in_channels, eps=1e-6, data_format="channels_first"),
                nn.Conv2d(in_channels, out_channels, kernel_size=(2,1), stride=(2,1)),
        )
        else:
            self.down = None

        self.residual = residual
        self.alpha = nn.Parameter(torch.zeros(1))
        
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                trunc_normal_(m.weight, std=.02)
                nn.init.constant_(m.bias, 0)
        

    def forward(self, x, preA=None):
        if preA!=None:
            A = self.PA
        else:
            A = self.PA
        input = x
        x = self.dwconv(x) # (N,C,T,V)
        x = x.permute(0, 2, 3, 1) # (N, C, T, V) -> (N, T, V, C)
        x = self.norm(x)
        x = self.pwconv1(x)#(N, T, V, 4*C)
        N,T,V,C4 = x.shape
        x = x.reshape(N,T,V,-1,4)
        x1 = x[:,:,:,:,0:3]
        x2 = x[:,:,:,:,3:4]
        if self.training:
            Ak = torch.unsqueeze(A,dim=0).repeat_interleave(N, dim=0)#.permute(0, 3, 2, 1).contiguous().view(-1,V,3)
            x1 = torch.einsum('niuk,ntkci->ntuci', Ak,x1)
        else:
            x1 = torch.einsum('iuk,ntkci->ntuci', A,x1)
        #x1 = torch.einsum('kuv,ntvck->ntuck', A,x1)
        x = torch.cat([x1,x2],dim=-1)
        x = x.reshape(N,T,V,C4)
        x = self.act(x)
        x = self.pwconv2(x)
        #if self.gamma is not None:
        #    x = self.gamma * x
        x = x.permute(0, 3, 1, 2) # (N, H, W, C) -> (N, C, H, W)
        if self.residual:
            y = input + self.drop_path(x)
        else:
            y = self.drop_path(x)

        if self.down!=None:
            y = self.down(y)

        return y,A



class Model(nn.Module):
    def __init__(self, num_class=2, num_point=25, num_person=9, graph=None, graph_args=dict(), in_channels=2,
                 drop_out=0, adaptive=True,unify=None):
        super(Model, self).__init__()

        self.graph = Graph(**graph_args)

        A = self.graph.A # 3,25,25
        if unify == "coco":
            from data.unifyposecode import COCO
            vindex = COCO
            A = A[:,vindex]
            A = A[:,:,vindex]
            num_point = len(vindex)
        elif unify == "ntu":
            from data.unifyposecode import NTU
            vindex = NTU
            A = A[:,vindex]
            A = A[:,:,vindex]
            num_point = len(vindex)
        
        self.num_class = num_class
        self.num_point = num_point
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * num_point)

        base_channel = 96
        dp_rates=[x.item() for x in torch.linspace(0, drop_out, 9)]
        stem = nn.Sequential(
            nn.Conv2d(in_channels,  base_channel, kernel_size=(4,1), stride=(4,1)),
            LayerNorm( base_channel, eps=1e-6, data_format="channels_first")
        )
        self.l1  = stem #TSGCNext_unit(in_channels, base_channel, A, residual=False, adaptive=adaptive,drop_path=dp_rates[0])
        # 2 5 2
        self.l2  = TSGCNext_unit(base_channel, base_channel, A, adaptive=adaptive,drop_path=dp_rates[0])
        self.l3  = TSGCNext_unit(base_channel, base_channel, A, adaptive=adaptive,drop_path=dp_rates[1])
        self.l4  = TSGCNext_unit(base_channel, base_channel, A, adaptive=adaptive,drop_path=dp_rates[2])
        self.l5  = TSGCNext_unit(base_channel, base_channel*2, A, adaptive=adaptive,drop_path=dp_rates[3])
        self.l6  = TSGCNext_unit(base_channel*2, base_channel*2, A, adaptive=adaptive,drop_path=dp_rates[4])
        self.l7  = TSGCNext_unit(base_channel*2, base_channel*2, A, adaptive=adaptive,drop_path=dp_rates[5])
        self.l8  = TSGCNext_unit(base_channel*2, base_channel*4, A, adaptive=adaptive,drop_path=dp_rates[6])
        self.l9  = TSGCNext_unit(base_channel*4, base_channel*4, A, adaptive=adaptive,drop_path=dp_rates[7])
        self.l10 = TSGCNext_unit(base_channel*4, base_channel*4, A, adaptive=adaptive,drop_path=dp_rates[8])
        
        self.norm = nn.LayerNorm(base_channel*4, eps=1e-6)
        #self.encode = nn.Linear(base_channel*4, base_channel*4)
        self.fc = nn.Linear(base_channel*4, num_class)
        
        self.in_channels = in_channels
        nn.init.normal_(self.fc.weight, 0, math.sqrt(2. / num_class))
    
        

    def forward(self, x):
        if len(x.shape) == 3:
            N, T, VC = x.shape
            x = x.view(N, T, self.num_point, -1).permute(0, 3, 1, 2).contiguous().unsqueeze(-1)
        if self.in_channels == 2:
            x = x[:,0:2,:,:,:]
        N, C, T, V, M = x.size()

        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        x = self.l1(x)
        x,A1 = self.l2(x)
        x,A2 = self.l3(x,A1)
        x,A3 = self.l4(x,A2)
        x,A4 = self.l5(x,A3)
        x,A5 = self.l6(x,A4)
        x,A6 = self.l7(x,A5)
        x,A7 = self.l8(x,A6)
        x,A8 = self.l9(x,A7)
        x,A9 = self.l10(x,A8)

        # N*M,C,T,V
        c_new = x.size(1)
        x = x.view(N, M, c_new, -1)
        x = x.mean(3).mean(1)
        x = self.norm(x)
        #x = self.encode(x)
        return self.fc(x)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

from tqdm import tqdm

def normalize_skeleton(X, width=1280, height=720):
    """
    X shape: (N, T, M, V, C)
    C = (x, y, confidence)
    """

    X = X.copy()

    # center theo hip (joint 8)
    hip = X[:, :, :, 8:9, :2]
    X[:, :, :, :, :2] = X[:, :, :, :, :2] - hip

    # normalize theo frame size
    X[:, :, :, :, 0] /= width
    X[:, :, :, :, 1] /= height

    # lấy confidence
    conf = X[:, :, :, :, 2:3]

    # confidence weighting
    X[:, :, :, :, 0:1] = X[:, :, :, :, 0:1] * conf
    X[:, :, :, :, 1:2] = X[:, :, :, :, 1:2] * conf

    return X
# ==================================
# DATASET CLASS
# ==================================

class MyDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.long)

        return x, y, idx


# ==================================
# METRICS (3 CLASS)
# ==================================

def print_metrics(y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)

    print("\nConfusion Matrix")
    print(cm)

    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="macro"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    print("\nMetrics")

    print(f"Accuracy : {acc*100:.2f}%")
    print(f"Precision: {precision*100:.2f}%")
    print(f"Recall   : {recall*100:.2f}%")
    print(f"F1-score : {f1*100:.2f}%")


# ==================================
# PROCESSOR
# ==================================

class Processor:

    def __init__(self, model, arg):

        self.arg = arg

        self.model = model.cuda()

        self.output_device = arg.device if isinstance(arg.device, int) else arg.device[0]

        self.best_acc = 0
        self.best_acc_epoch = 0
        self.start_epoch = arg.start_epoch

        os.makedirs(arg.work_dir, exist_ok=True)

        # ==================================
        # LOAD DATASET
        # ==================================

        print("Loading dataset...")

        # =====================
        # TRAIN
        # =====================
        
        X_train = np.load(
            '../X_train.npy'
        )
        
        y_train = np.load(
            '../y_train.npy'
        )
        
        # =====================
        # VAL
        # =====================
        
        X_val = np.load(
            '../X_val.npy'
        )
        
        y_val = np.load(
            '../y_val.npy'
        )
        
        # =====================
        # TEST
        # =====================
        
        test_X = np.load(
            '../X_test.npy'
        )
        
        test_y = np.load(
            '../y_test.npy'
        )
        print("Normalizing skeleton data...")

        X_train = normalize_skeleton(X_train, 1280, 720)
        X_val   = normalize_skeleton(X_val, 1280, 720)
        test_X  = normalize_skeleton(test_X, 1280, 720)
        # =====================
        # TRANSPOSE
        # (N,T,M,V,C) → (N,C,T,V,M)
        # =====================
        
        X_train = np.transpose(X_train, (0,4,1,3,2))
        X_val   = np.transpose(X_val,   (0,4,1,3,2))
        test_X  = np.transpose(test_X,  (0,4,1,3,2))
        print("Train:", X_train.shape)
        print("Val:", X_val.shape)
        print("Test:", test_X.shape)

        self.data_loader = {

            'train': DataLoader(
                MyDataset(X_train, y_train),
                batch_size=arg.batch_size,
                shuffle=True,
                num_workers=2
            ),

            'val': DataLoader(
                MyDataset(X_val, y_val),
                batch_size=arg.test_batch_size,
                shuffle=False,
                num_workers=2
            ),

            'test': DataLoader(
                MyDataset(test_X, test_y),
                batch_size=arg.test_batch_size,
                shuffle=False,
                num_workers=2
            )
        }

        self.loss = nn.CrossEntropyLoss().cuda(self.output_device)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=arg.base_lr,
            weight_decay=arg.weight_decay
        )

        resume_path = os.path.join(
            arg.work_dir,
            "latest_checkpoint.pt"
        )

        if os.path.exists(resume_path):

            checkpoint = torch.load(
                resume_path,
                weights_only=False
            )

            self.model.load_state_dict(
                checkpoint['model_state']
            )

            self.optimizer.load_state_dict(
                checkpoint['optim_state']
            )

            self.best_acc = checkpoint['best_acc']
            self.best_acc_epoch = checkpoint['best_acc_epoch']

            self.start_epoch = checkpoint['epoch'] + 1

            print(
                f"Resumed from epoch {self.start_epoch}"
            )

        else:

            print("Training from scratch")


    def train(self, epoch):

        self.model.train()

        print(f"\nEpoch {epoch+1} Training")

        loader = self.data_loader['train']

        loss_value = []
        acc_value = []

        for data, label, _ in tqdm(loader, ncols=60):

            data = data.cuda()
            label = label.cuda()

            output = self.model(data)

            loss = self.loss(output, label)

            self.optimizer.zero_grad()

            loss.backward()

            self.optimizer.step()

            _, pred = torch.max(output, 1)

            acc = torch.mean(
                (pred == label).float()
            )

            loss_value.append(loss.item())
            acc_value.append(acc.item())

        print(
            f"Train loss: {np.mean(loss_value):.4f}"
        )

        print(
            f"Train acc : {np.mean(acc_value)*100:.2f}%"
        )

    def eval(self, epoch, mode='val'):

        self.model.eval()

        loader = self.data_loader[mode]

        loss_value = []
        score_frag = []
        label_list = []

        with torch.no_grad():

            for data, label, _ in tqdm(loader, ncols=60):

                data = data.cuda()
                label = label.cuda()

                output = self.model(data)

                loss = self.loss(output, label)

                loss_value.append(loss.item())

                score_frag.append(
                    output.cpu().numpy()
                )

                label_list.append(
                    label.cpu().numpy()
                )

        score = np.concatenate(score_frag)

        label_list = np.concatenate(label_list)

        acc = accuracy_score(
            label_list,
            np.argmax(score, axis=1)
        )

        print(
            f"{mode} loss: {np.mean(loss_value):.4f}"
        )

        print(
            f"{mode} acc : {acc*100:.2f}%"
        )

        if mode == 'val' and acc > self.best_acc:

            self.best_acc = acc
            self.best_acc_epoch = epoch + 1

            torch.save(
                self.model.state_dict(),
                os.path.join(
                    self.arg.work_dir,
                    'best_model.pt'
                )
            )

            print("New best model saved")

        return acc

    def start(self):

        for epoch in range(
            self.start_epoch,
            self.arg.num_epoch
        ):

            self.train(epoch)

            self.eval(epoch, 'val')

            torch.save({

                'epoch': epoch,

                'model_state': self.model.state_dict(),

                'optim_state': self.optimizer.state_dict(),

                'best_acc': self.best_acc,

                'best_acc_epoch': self.best_acc_epoch

            },

            os.path.join(
                self.arg.work_dir,
                "latest_checkpoint.pt"
            ))

        print(
            f"\nBest val acc {self.best_acc*100:.2f}% "
            f"at epoch {self.best_acc_epoch}"
        )
    def test_best(self):

        print("\nEvaluating BEST model")

        best_model_path = os.path.join(
            self.arg.work_dir,
            'best_model.pt'
        )

        self.model.load_state_dict(
            torch.load(best_model_path)
        )

        self.model.eval()

        all_labels = []
        all_preds = []

        with torch.no_grad():

            for data, label, _ in self.data_loader['test']:

                data = data.cuda()
                label = label.cuda()

                output = self.model(data)

                _, pred = torch.max(output, 1)

                all_labels.append(
                    label.cpu().numpy()
                )

                all_preds.append(
                    pred.cpu().numpy()
                )

        all_labels = np.concatenate(all_labels)
        all_preds = np.concatenate(all_preds)

        print_metrics(
            all_labels,
            all_preds
        )

In [ ]:
class Args:
    device = 0
    batch_size = 16
    test_batch_size = 32
    base_lr = 0.0001
    weight_decay = 1e-5
    num_epoch = 100
    start_epoch = 0
    save_interval = 15
    model_saved_name = 'my_model'
    work_dir = './results'

arg = Args()

model = Model(
    num_class=7,
    num_point=25,
    num_person=9,
    in_channels=2,
    graph=Graph,
    graph_args={'labeling_mode': 'spatial'}
)

In [ ]:
processor = Processor(model, arg)
processor.start()   
processor.test_best()